# PD05 · Particionamiento y comparación pandas–Polars

Continuación de PD01–PD03 basada en las clases del 9 y 10 de octubre de 2026. Tiempo orientativo: 180 minutos, con el entorno y los datos preparados.

Copia esta plantilla a `soluciones/` conservando su nombre. Escribe allí el código y las interpretaciones. Las celdas iniciales muestran las lecturas; las actividades quedan a tu cargo. Antes de entregar, reinicia el kernel y ejecuta todas las celdas en orden.

Utiliza directamente las API de pandas y, en PD05, Polars. Mantén visibles las conversiones, máscaras, agrupaciones, uniones y exportaciones. No uses `talleres.py`, `pqrs_talleres.py`, comandos de shell ni funciones que resuelvan varias etapas sin mostrar su implementación. Un bucle de medición es válido si su cuerpo contiene las operaciones que estás midiendo. No instales paquetes desde el notebook.

[Índice](../ejercicios/README.md) · [Pandas](../docs/manual-pandas.md) · [Polars](../docs/manual-polars.md)

## Ubicación de trabajo

Abre Jupyter desde la raíz del repositorio con el entorno del curso activo. Cambia `REPO` por la ruta real de tu copia. Las rutas son explícitas y funcionan tanto desde `ejercicios/` como desde `soluciones/`. Si falta un archivo, revisa su ubicación; no sustituyas silenciosamente la entrada por otro dataset.

## Pregunta y preparación

¿Cómo cambian lectura, escritura y procesamiento al variar formato, particionamiento y motor?

Requiere el Parquet completo de PD04 y las dependencias del entorno del curso: `requirements.txt` y `requirements_pqrs.txt`. Trabaja en CPU. No dupliques filas para simular volumen. Esta actividad estudia los resultados de este equipo y estos datos.

Las lecturas siguientes son directas. Para comparar CSV más adelante, escribe explícitamente `pd.read_csv` y `pl.read_csv`, sus separadores, política de nulos y tipos. No uses inferencia diferente de tipos como si fuera trabajo equivalente.

In [ ]:
import os
import time
import platform
import pandas as pd
import polars as pl
import pyarrow as pa

REPO = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
INPUT = REPO + "/kit/salidas/soluciones/PD04/analisis_integrados.parquet"
OUT = REPO + "/kit/salidas/soluciones/PD05"
os.makedirs(OUT, exist_ok=True)
print("Entrada:", INPUT)
print("Salidas:", OUT)
print(platform.platform())
print("pandas:", pd.__version__, "Polars:", pl.__version__, "PyArrow:", pa.__version__)

In [ ]:
datos_pd = pd.read_parquet(INPUT, engine="pyarrow")
display(datos_pd.head())
print(datos_pd.shape)
print(datos_pd.dtypes)

In [ ]:
datos_pl = pl.read_parquet(INPUT)
print(datos_pl.head())
print(datos_pl.shape)
print(datos_pl.schema)

## 1. Contrato común y consulta · 25 minutos

Proyecta en ambos motores las mismas columnas: `id_fila`, `anio`, `codigo_departamento`, `codigo_municipio`, `ph` y `ph_valido`. Mantén todas las filas. Declara tipos equivalentes, cómo representar nulos en CSV y cómo restaurar códigos con ceros iniciales.

Elige un año y departamento presentes. Filtra y agrupa por municipio para obtener total de análisis, cantidad de pH válidos y media del pH válido. Conserva el grupo municipal ausente, si existe, y la media ausente cuando no hay pH válidos. En pandas utiliza `groupby(..., dropna=False)`; en Polars comprueba el tratamiento equivalente de claves nulas.

**Control:** compara claves y conteos exactos; medias con tolerancia absoluta 0.000001 y relativa cero. Ordena antes de comparar y comprueba también dónde aparecen nulos.

In [ ]:
# Desarrollo
# Prepara la proyección común e implementa la consulta en ambos motores.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## 2. Escritura y particionamiento desde pandas · 35 minutos

Escribe la proyección completa como CSV, Parquet único y dos datasets Parquet: particionado solo por año y particionado por año y departamento. Usa PyArrow, compresión Snappy e `index=False` para Parquet.

Para las carpetas define `anio_particion` y `departamento_particion` como columnas auxiliares de texto. Usa `SIN_DATO` para claves ausentes, después de comprobar que no coincide con valores reales. Conserva además las columnas originales tipadas. Incorpora estas auxiliares a todas las representaciones que compares, de modo que tengan el mismo contenido.

Muestra llamadas directas a `to_csv` y `to_parquet`; este último admite `partition_cols=["anio_particion", "departamento_particion"]`. Ejemplo de estructura: `por_anio_departamento/anio_particion=2024/departamento_particion=15/…parquet`.

**Control:** inventaría filas, archivos y bytes por partición, incluyendo SIN_DATO. Relee y concilia con el total de entrada. Compara ambos diseños y justifica si hay demasiados archivos pequeños. Particionar archivos no implica procesamiento paralelo en pandas.

In [ ]:
# Desarrollo
# Escribe las cuatro representaciones y presenta su inventario de particiones.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## 3. Lectura y escritura comparables · 35 minutos

Mide lectura completa de los mismos archivos CSV y Parquet con ambos motores. Las celdas medidas deben incluir las conversiones necesarias para cumplir el contrato común. Mide por separado escritura de CSV y Parquet desde los datos equivalentes ya residentes en memoria; no incluyas la carga en ese tiempo. Usa el mismo orden de columnas, separador, nulos y compresión, y documenta opciones distintas que no puedas igualar.

Usa `time.perf_counter()` inmediatamente antes y después de cada operación. Realiza una ejecución inicial separada y cinco repeticiones, alternando el orden de motores. Escribe cada repetición en una ruta nueva; no acumules archivos dentro de un dataset particionado ya existente. Deja impresión, validación e inventario fuera del intervalo medido.

**Control:** relee los resultados fuera de la medición y verifica equivalencia. Conserva tiempos individuales y mediana. Documenta versiones, CPU, RAM disponible y ubicación de archivos. No describas una lectura como fría sin haber controlado la caché del sistema.

In [ ]:
# Desarrollo
# Mide con llamadas directas a read_csv/read_parquet y to_csv/to_parquet o write_csv/write_parquet.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## 4. Procesamiento y consulta desde disco · 40 minutos

Separa dos experimentos:

1. **En memoria:** parte de las tablas ya cargadas y mide solo filtro y agregación en pandas y Polars eager.
2. **Desde disco:** incluye lectura, filtro y agregación; compara Parquet único, particionado por año y particionado por año/departamento. En pandas muestra `pd.read_parquet` con `columns` y `filters`. En Polars muestra `pl.scan_parquet`, selección, filtro, agregación y `collect()` dentro del tiempo medido. Para particiones Hive declara `hive_partitioning=True` y el esquema de las claves auxiliares como texto para preservar ceros y SIN_DATO.

En archivos particionados usa filtros sobre las claves de partición y aplica también las condiciones analíticas correspondientes. En el archivo único aplica una selección equivalente. Usa las mismas cinco repeticiones y alterna los motores. Muestra `explain()` fuera del tiempo medido para discutir qué operaciones puede evitar Polars.

**Control:** todas las variantes deben devolver la misma consulta. No compares construir un plan diferido con ejecutar una agregación. Reporta las particiones compatibles con el filtro; no las describas como bytes realmente leídos sin instrumentación. No mezcles el efecto del formato, la partición y el motor en una sola explicación.

**Tipos de las carpetas:** PyArrow puede inferir `2024` y `15` como enteros aunque las columnas auxiliares fueran texto. Declara el esquema Hive de forma explícita para las lecturas de pandas; así los filtros usan el mismo tipo que las particiones. Ejemplo para el diseño de dos claves (ajusta la ruta):

```python
import pyarrow.dataset as ds

particion_hive = ds.partitioning(
    pa.schema([
        ("anio_particion", pa.string()),
        ("departamento_particion", pa.string()),
    ]),
    flavor="hive"
)
seleccion_pd = pd.read_parquet(
    OUT + "/por_anio_departamento",
    engine="pyarrow",
    partitioning=particion_hive,
    columns=["id_fila", "codigo_municipio", "ph", "ph_valido"],
    filters=[("anio_particion", "=", "2024"),
             ("departamento_particion", "=", "15")]
)
```

Sustituye año y departamento por tu selección. Para el diseño por año declara únicamente `anio_particion` y aplica el departamento como filtro sobre su columna original. Esta configuración de PyArrow solo declara cómo leer las carpetas; el análisis sigue visible en pandas.


In [ ]:
# Desarrollo
# Ejecuta los dos experimentos y comprueba resultados materializados equivalentes.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## 5. Reejecución, evidencia y decisión · 45 minutos

Repite la construcción de los archivos en otra carpeta de ejecución nueva. No borres carpetas de datos originales ni reutilices un directorio particionado con archivos viejos. Comprueba contenido y total de registros; el nombre de los archivos o su hash no tiene por qué coincidir para que el contenido sea equivalente.

Guarda `mediciones.csv` con motor, versión, operación, formato, particionamiento, repetición, segundos, filas de entrada y filas de salida. Guarda aparte la ejecución inicial; calcula la mediana solo con las cinco repeticiones medidas. Añade inventario de bytes/archivos y controles de equivalencia.

Responde: ¿qué parte mejora con Parquet?, ¿cuándo ayuda la selección de particiones?, ¿qué diseño genera archivos demasiado pequeños?, ¿qué cambia entre tiempo en memoria y tiempo desde disco?, ¿qué elegirías para este conjunto y por qué?

**Control:** no presupongas que Polars o el particionamiento ganarán. Diferencias pequeñas o variables deben describirse como tales. El tamaño de un DataFrame no equivale a la memoria máxima del proceso.

In [ ]:
# Desarrollo
# Concilia la segunda ejecución, exporta las mediciones y redacta la recomendación.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## Entrega y referencias de API

`soluciones/PD05_particionamiento_pandas_polars.ipynb`, mediciones, inventario y controles regenerables en `kit/salidas/soluciones/PD05/`. Incluye una recomendación basada en resultados; no se califica obtener el menor tiempo.

- [pandas: to_parquet y partition_cols](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html)
- [Polars: fuentes y ejecución diferida](https://docs.pola.rs/user-guide/lazy/sources_sinks/)
- [Polars: scan_parquet](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_parquet.html)

Usa las versiones fijadas por el curso; si la documentación más reciente muestra una opción diferente, comprueba su disponibilidad antes de usarla.